## `COPY`, `ADD` & the build context

`RUN` builds things; **`COPY` and `ADD` bring your files into the image** — but they can only see files inside the *build context*, so those three ideas belong together.

**`COPY <src> <dst>`** copies files or directories from the context. That's it — no magic, predictable:

```dockerfile
COPY requirements.txt /app/
COPY . /app/
```

**`ADD`** is `COPY` plus two extras: if `<src>` is a **URL** it downloads it, and if `<src>` is a local **tarball** it auto-extracts it. Both are footguns — the auto-extract is surprising, the URL form bypasses cache discipline. Best practice:

> **Use `COPY` by default. Use `ADD` only for the auto-extract case; use `RUN curl` instead of `ADD <URL>`.**

Both honour `--chown=user:group` and `--from=<stage>` (multi-stage, later).

**The build context.** In `docker build -t myimage .`, that final `.` is the context — and *the whole thing is tarballed and shipped to the daemon before the build starts.* Two consequences most people miss:

- If the context is `~/projects` with 12 GB of `node_modules`, you wait minutes for the upload even if the Dockerfile copies one file. **Set the context narrowly** — build from the project root, not `~`.
- `COPY` can only see *inside* the context. `COPY ../shared/lib /app/` does **not** work — anything above the context root is invisible.

**`.dockerignore`** trims that tarball. A `.gitignore`-style file at the context root, one pattern per line:

```
.git
node_modules
__pycache__
*.pyc
.env
.env.*
!README.md
```

Three payoffs: **faster builds** (smaller context), **better cache hits** (a `COPY . /app/` layer isn't busted by unrelated editor files), and **security** — it stops you baking `.env` secrets or `.git/` history into a public image. Treat it as part of every Dockerfile, not optional.